# Demo: Transfer Learning e Fine-Tuning na Prática (Gabarito)
**Disciplina:** Inteligência Artificial e Machine Learning Avançado
**Contexto:** Análise de Sentimento de Textos

Neste notebook, vamos desmistificar o processo de Transfer Learning em Modelos de Linguagem (NLP).
Historicamente, treinar um modelo de linguagem do zero exige terabytes de texto e milhões de dólares em GPUs. Hoje, vamos pegar um modelo pré-treinado na Wikipedia (DistilBERT) e adaptá-lo para uma tarefa de Análise de Sentimento.

**Nossa jornada hoje:**
1. Carregar um modelo pré-treinado (A "Mente" Geral).
2. **Feature Extraction:** Usar o modelo apenas como extrator de características (congelando seu conhecimento).
3. **Fine-Tuning:** Descongelar o modelo e ajustá-lo especificamente para nossa tarefa.

In [1]:
# Instalação das bibliotecas necessárias (descomente se estiver rodando no Google Colab)
# !pip install -q transformers datasets scikit-learn torch

import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Configurando o dispositivo (GPU se disponível, senão CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Rodando em: {device}")

Rodando em: cpu


## 1. O Dataset e o Tokenizador
Vamos usar um dataset clássico e leve de análise de sentimento (Rotten Tomatoes - críticas de filmes) contendo frases curtas classificadas como Positivas (1) ou Negativas (0).
O Tokenizador é responsável por transformar nossas palavras nos números que o modelo entende.

In [11]:
# Carregando um dataset leve para a demonstração (apenas 2000 exemplos para ser rápido)
dataset = load_dataset("rotten_tomatoes")
train_data = dataset["train"].shuffle(seed=42).select(range(2000))
test_data = dataset["test"].shuffle(seed=42).select(range(500))

# Carregando o tokenizador do DistilBERT
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

# Função para tokenizar o texto
def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)

# Aplicando a tokenização ao dataset
train_encoded = train_data.map(tokenize, batched=True, batch_size=None)
test_encoded = test_data.map(tokenize, batched=True, batch_size=None)

# Convertendo para tensores PyTorch
train_encoded.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_encoded.set_format("torch", columns=["input_ids", "attention_mask", "label"])

/Users/dhenyfernandes/anaconda3/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

## 2. Abordagem 1: Feature Extraction (Congelando a Rede)
Nesta abordagem de Transfer Learning, nós **não alteramos os pesos** do modelo original. Nós passamos nossos textos por ele, extraímos o "entendimento" do modelo (chamado de *Hidden State* ou *Embedding*) e treinamos um classificador clássico (como Regressão Logística) por cima.

### Exercício 1: Congelando os pesos (Gabarito)
Para garantir que o modelo não aprenda nada novo nesta etapa e funcione apenas como um extrator estático, precisamos congelar seus parâmetros iterando sobre eles e desativando o cálculo de gradientes.

In [12]:
# Carregando o modelo base (sem a camada de classificação final)
model_extractor = AutoModel.from_pretrained(model_ckpt).to(device)

# ==========================================
# GABARITO - EXERCÍCIO 1: Congelando os pesos
# ==========================================
for param in model_extractor.parameters():
    param.requires_grad = False
# ==========================================

# Função para extrair os embeddings da última camada do modelo
def extract_hidden_states(batch):
    inputs = {k: v.to(device) for k, v in batch.items() if k in tokenizer.model_input_names}
    with torch.no_grad():
        last_hidden_state = model_extractor(**inputs).last_hidden_state
    # Pegamos o vetor correspondente ao token [CLS] (primeira posição), que resume a frase
    return {"hidden_state": last_hidden_state[:, 0].cpu().numpy()}

# Extraindo as features (isso pode levar alguns segundos)
print("Extraindo features...")
train_features = train_encoded.map(extract_hidden_states, batched=True, batch_size=16)
test_features = test_encoded.map(extract_hidden_states, batched=True, batch_size=16)

X_train = np.array(train_features["hidden_state"])
y_train = np.array(train_features["label"])
X_test = np.array(test_features["hidden_state"])
y_test = np.array(test_features["label"])

Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing DistilBertModel: ['vocab_transform.weight', 'vocab_layer_norm.bias', 'vocab_layer_norm.weight', 'vocab_transform.bias', 'vocab_projector.bias']
- This IS expected if you are initializing DistilBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DistilBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Extraindo features...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

### Exercício 2: Treinando o "Head" de Classificação (Gabarito)
Agora que transformamos nossos textos em matrizes numéricas ricas em contexto (embeddings), precisamos treinar um modelo simples em cima delas.

In [14]:
# ==========================================
# GABARITO - EXERCÍCIO 2: Treinar Classificador
# ==========================================
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Acurácia com Feature Extraction: {accuracy:.4f}")
# ==========================================

Acurácia com Feature Extraction: 0.7900


## 3. Abordagem 2: Fine-Tuning (Descongelando a Rede)
Na extração de features, a acurácia foi boa, mas o modelo não adaptou seu "vocabulário" interno para a nossa tarefa específica.

No **Fine-Tuning**, nós substituímos o classificador final e **deixamos o erro (gradiente) fluir por toda a rede**. Assim, o modelo altera seus pesos internos.

### Exercício 3: O Segredo do Fine-Tuning (Taxa de Aprendizado) (Gabarito)
Se usarmos uma Taxa de Aprendizado (*Learning Rate*) muito alta, nós destruiremos os pesos que o modelo aprendeu. Por isso, usamos valores muito baixos.

In [15]:
# Carregamos o modelo já com uma camada de classificação final não-treinada
model_finetune = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=2).to(device)

# ==========================================
# GABARITO - EXERCÍCIO 3: Taxa de Aprendizado
# Em fine-tuning, geralmente usamos valores entre 2e-5 e 5e-5.
# ==========================================
learning_rate_escolhido = 2e-5
# ==========================================

# Configurações do treinamento
training_args = TrainingArguments(
    output_dir="./resultados",
    learning_rate=learning_rate_escolhido,
    num_train_epochs=2, # Apenas 2 épocas são suficientes com Fine-Tuning!
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    evaluation_strategy="epoch"
)

# Inicializando o Trainer da HuggingFace
trainer = Trainer(
    model=model_finetune,
    args=training_args,
    train_dataset=train_encoded,
    eval_dataset=test_encoded
)

print("Iniciando o Fine-Tuning...")
trainer.train()

# Avaliando o resultado
results = trainer.evaluate()
print(f"Resultado do Fine-Tuning: {results}")

/Users/dhenyfernandes/anaconda3/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing DistilBertForSequenceClassification: ['vocab_transform.weight', 'vocab_layer_norm.bias', 'vocab_layer_norm.weight', 'vocab_transform.bias', 'vocab_projector.bias']
- This IS expected if you are initializing DistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DistilBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequ

Iniciando o Fine-Tuning...


***** Running training *****
  Num examples = 2000
  Num Epochs = 2
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 250
  Number of trainable parameters = 66955010


Epoch,Training Loss,Validation Loss
1,No log,0.426437
2,No log,0.433654


The following columns in the evaluation set don't have a corresponding argument in `DistilBertForSequenceClassification.forward` and have been ignored: text. If text are not expected by `DistilBertForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Evaluation *****
  Num examples = 500
  Batch size = 16
The following columns in the evaluation set don't have a corresponding argument in `DistilBertForSequenceClassification.forward` and have been ignored: text. If text are not expected by `DistilBertForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Evaluation *****
  Num examples = 500
  Batch size = 16


Training completed. Do not forget to share your model on huggingface.co/models =)


The following columns in the evaluation set don't have a corresponding argument in `DistilBertForSequenceClassification.forward` and have been ignored: text. If text are not expected by `DistilBertForSequenceClassification.forward`,  

Resultado do Fine-Tuning: {'eval_loss': 0.4336535930633545, 'eval_runtime': 27.6254, 'eval_samples_per_second': 18.099, 'eval_steps_per_second': 1.158, 'epoch': 2.0}


In [16]:
# Faz as predições no dataset de teste
predicoes_brutas = trainer.predict(test_encoded)

# Extrai as classes previstas (0 ou 1)
y_pred_finetune = np.argmax(predicoes_brutas.predictions, axis=-1)

# Pega nas labels reais
y_true_finetune = predicoes_brutas.label_ids

# Calcula a acurácia
acuracia_final = accuracy_score(y_true_finetune, y_pred_finetune)
print(f"Acurácia do Fine-Tuning: {acuracia_final:.4f}")

The following columns in the test set don't have a corresponding argument in `DistilBertForSequenceClassification.forward` and have been ignored: text. If text are not expected by `DistilBertForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Prediction *****
  Num examples = 500
  Batch size = 16


Acurácia do Fine-Tuning: 0.8120


### Conclusão da Demo
* Com **Feature Extraction**, treinamos um modelo em segundos, pois a rede profunda estava congelada. Excelente quando temos poucos recursos.
* Com **Fine-Tuning**, gastamos mais tempo e processamento, mas o modelo se adaptou profundamente aos dados, geralmente superando o Feature Extraction em acurácia, especialmente em domínios de nicho (como finanças ou direito).